# Basic RAG

In this module we will see the basic architecture for a RAG. In code, it looks something like this:

```python
    def llm(prompt):
        response = openai_client.responses.create(
            model="gpt-5.4-mini",
            input=prompt
        )
        return response.output_text


    def rag(question):
        search_results = search(question)
        user_prompt = build_prompt(question, search_results)
        return llm(user_prompt)
```

The user will ask a question. We will retrieve the most relevant pieces of information from our dataset, and we will build a user prompt which will be the original question enriched with the relevant context from our dataset. Finally, we will send the question to the API, which will return the correct answer.

## Imports

In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

## Dataset

For the course, we're going to use the FAQ from DataTalks.Club as our example dataset: we want a bot that can answer student questions by retrieving and parsing the relevant parts of the dataset.

This dataset contains all of the frequently asked questions in the DataTalks.Club Slack channel, and is available at a JSON endpoint we can fetch directly.

In [2]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 103},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

The main endpoint gives us the path of the FAQ data for that specific course; let's fetch all the FAQ documents for all courses.

In [3]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1368

In [4]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

Each entry has:

- `id`: unique identifier for the entry
- `course`: course slug
- `section`: which section of the course the entry belongs to
- `question`: the frequently answered question
- `answer`: the frequently answered answer

In [5]:
documents[684]

{'id': 'cf068009f1',
 'course': 'mlops-zoomcamp',
 'section': 'Module 1: Introduction',
 'question': 'AWS EC2: How do I handle changing IP addresses on restart?',
 'answer': 'Every time I restart my EC2 instance, I receive a different IP and need to update the config file manually.\n\n**Solution:**\n\nYou can create a script to automatically update the IP address of your EC2 instance. Refer to this [guide](https://github.com/dimzachar/mlops-zoomcamp/blob/master/notes/Week_1/update_ssh_config.md) for detailed steps.'}

This dataset is our knowledge base for our project:

1. We index all the documents
2. When a student asks a question, we search the index
3. The search returns the most relevant FAQ entries
4. We give those entries to the LLM as context
5. The LLM generates an answer based on the context

## Indexing and searching the dataset

We're going to use [minsearch](https://github.com/alexeygrigorev/minsearch) for this step, because it's lightweight and still does the job well for our current scope (a toy RAG). It basically counts word importance using TF-IDF and ranks documents by that score. Keyword-based indexing, no meaning, just math on word frequencies.

In this case, we'll index the `section`, `question` and `answer` fields as text (tokenized and ranked). The `course` field will be indexed as keyword for filtering.

In [6]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

Let's try a search with the question we used before, in the `01_RAG_intro` notebook:

In [7]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [8]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
 'How should I start the course and follow the weekly workflow?']

The search returned five relevant questions from the dataset: questions about joining the course, registration, certificates. These are the documents that we will send to the LLM.

Let's wrap this in a `search` function, the first component of our RAG pipeline:

In [9]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

## Building the prompt

We need to build a prompt that includes the user's question and the search result.

The prompt to send to the API will be composed of two main parts:

- `INSTRUCTIONS`: this tells the LLM how to behave. It never changes, so it's the same for every request.
- `USER PROMPT`: this changes with every request; it's the actual question and the retrieved context.

In [10]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [11]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [12]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [13]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [14]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [15]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [16]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=prompt
)

In [17]:
response.output_text

'Yes — you can join now.\n\nYou can start whenever you want, but if you want a certificate, you need to submit your project while submissions are still open.'

In [18]:
response.output[0].content[0].text

'Yes — you can join now.\n\nYou can start whenever you want, but if you want a certificate, you need to submit your project while submissions are still open.'

In [19]:
response.usage

ResponseUsage(input_tokens=678, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=37, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=715)

In [20]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.000675

In [21]:
message_history = [
    {'role': 'developer', 'content': INSTRUCTIONS},
    {'role': 'user', 'content': prompt}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=message_history
)

In [22]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [23]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [24]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join now. If you want a certificate, make sure to submit your project while submissions are still open.


In [25]:
rag("How do I get a certificate?")

'You can only get a certificate if you finish the course with a live cohort and pass the capstone project. Self-paced mode does not award certificates because you need to peer-review 3 capstones while the course is running.'